<a href="https://colab.research.google.com/github/kevinmullet/olist-analysis/blob/main/olist_customers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import os
import glob
import sqlite3

In [2]:
FOLDER_PATH = '/content/drive/MyDrive/SQL Project'
#https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce?select=olist_order_payments_dataset.csv

conn = sqlite3.connect(f'{FOLDER_PATH}/olist.db')

for f in glob.glob(f'{FOLDER_PATH}/*.csv'): #
    name = os.path.basename(f).replace('olist_', '').replace('_dataset', '').replace('.csv', '') # merapikan format nama csv
    pd.read_csv(f).to_sql(name, conn, if_exists='replace', index=False)

In [3]:
pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)

,name
0,customers
1,geolocation
2,order_items
3,order_payments
4,order_reviews
5,orders
6,products
7,sellers
8,product_category_name_translation


In [4]:
def q(query): #function untuk shortcut
  return pd.read_sql(query,conn)

In [5]:
#Berapa total order di database ini?

q("""
SELECT
COUNT(*) AS total_orders
FROM orders
""")

,total_orders
0,99441


In [6]:
#Ada berapa order untuk tiap status (delivered, canceled, dst.)?
q("SELECT * FROM ORDERS")

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00
...,...,...,...,...,...,...,...,...
99436,9c5dedf39a927c1b2549525ed64a053c,39bd1228ee8140590ac3aca26f2dfe00,delivered,2017-03-09 09:54:05,2017-03-09 09:54:05,2017-03-10 11:18:03,2017-03-17 15:08:01,2017-03-28 00:00:00
99437,63943bddc261676b46f01ca7ac2f7bd8,1fca14ff2861355f6e5f14306ff977a7,delivered,2018-02-06 12:58:58,2018-02-06 13:10:37,2018-02-07 23:22:42,2018-02-28 17:37:56,2018-03-02 00:00:00
99438,83c1379a015df1e13d02aae0204711ab,1aa71eb042121263aafbe80c1b562c9c,delivered,2017-08-27 14:46:43,2017-08-27 15:04:16,2017-08-28 20:52:26,2017-09-21 11:24:17,2017-09-27 00:00:00
99439,11c177c8e97725db2631073c19f07b62,b331b74b18dc79bcdf6532d51e1637c1,delivered,2018-01-08 21:28:27,2018-01-08 21:36:21,2018-01-12 15:35:03,2018-01-25 23:32:54,2018-02-15 00:00:00


In [7]:
q("""
SELECT
  COUNT(*) AS number_of_orders,
  order_status
FROM orders
GROUP BY order_status
""")

,number_of_orders,order_status
0,2,approved
1,625,canceled
2,5,created
3,96478,delivered
4,314,invoiced
5,301,processing
6,1107,shipped
7,609,unavailable


In [8]:
#Negara bagian mana yang punya pelanggan paling banyak? Tampilkan 5 teratas.
q("""
SELECT *
FROM customers
""")

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP
...,...,...,...,...,...
99436,17ddf5dd5d51696bb3d7c6291687be6f,1a29b476fee25c95fbafc67c5ac95cf8,3937,sao paulo,SP
99437,e7b71a9017aa05c9a7fd292d714858e8,d52a67c98be1cf6a5c84435bd38d095d,6764,taboao da serra,SP
99438,5e28dfe12db7fb50a4b2f691faecea5e,e9f50caf99f032f0bf3c55141f019d99,60115,fortaleza,CE
99439,56b18e2166679b8a959d72dd06da27f9,73c2643a0a458b49f58cea58833b192e,92120,canoas,RS


In [9]:
q("""
SELECT
  customer_state,
  COUNT(DISTINCT(customer_unique_id)) AS number_of_customers
FROM customers
GROUP BY customer_state
ORDER BY number_of_customers DESC
LIMIT 5
""")

,customer_state,number_of_customers
0,SP,40302
1,RJ,12384
2,MG,11259
3,RS,5277
4,PR,4882


In [10]:
#Revenue per metode pembayaran
q("""
SELECT
*
FROM order_payments
""")

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45
...,...,...,...,...,...
103881,0406037ad97740d563a178ecc7a2075c,1,boleto,1,363.31
103882,7b905861d7c825891d6347454ea7863f,1,credit_card,2,96.80
103883,32609bbb3dd69b3c066a6860554a77bf,1,credit_card,1,47.77
103884,b8b61059626efa996a60be9bb9320e10,1,credit_card,5,369.54


In [11]:
q("""
SELECT
  payment_type,
  ROUND(SUM(payment_value),2) AS total_revenue
FROM order_payments
GROUP BY payment_type
ORDER BY total_revenue DESC
""")

,payment_type,total_revenue
0,credit_card,12542084.19
1,boleto,2869361.27
2,voucher,379436.87
3,debit_card,217989.79
4,not_defined,0.00


In [12]:
#Top 10 kategori berdasarkan revenue
q("""
SELECT *
FROM products
""")

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0
...,...,...,...,...,...,...,...,...,...
32946,a0b7d5a992ccda646f2d34e418fff5a0,moveis_decoracao,45.0,67.0,2.0,12300.0,40.0,40.0,40.0
32947,bf4538d88321d0fd4412a93c974510e6,construcao_ferramentas_iluminacao,41.0,971.0,1.0,1700.0,16.0,19.0,16.0
32948,9a7c6041fa9592d9d9ef6cfe62a71f8c,cama_mesa_banho,50.0,799.0,1.0,1400.0,27.0,7.0,27.0
32949,83808703fc0706a22e264b9d75f04a2e,informatica_acessorios,60.0,156.0,2.0,700.0,31.0,13.0,20.0


In [13]:
q("""
SELECT *
FROM product_category_name_translation
""")

,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor
...,...,...
66,flores,flowers
67,artes_e_artesanato,arts_and_craftmanship
68,fraldas_higiene,diapers_and_hygiene
69,fashion_roupa_infanto_juvenil,fashion_childrens_clothes


In [14]:
q("""
SELECT
  pt.product_category_name_english AS category,
  ROUND(SUM(oi.price),2) AS revenue
FROM products p
JOIN order_items oi ON p.product_id = oi.product_id
JOIN product_category_name_translation pt ON p.product_category_name = pt.product_category_name
GROUP BY category
ORDER BY revenue DESC
LIMIT 10
""")

,category,revenue
0,health_beauty,1258681.34
1,watches_gifts,1205005.68
2,bed_bath_table,1036988.68
3,sports_leisure,988048.97
4,computers_accessories,911954.32
5,furniture_decor,729762.49
6,cool_stuff,635290.85
7,housewares,632248.66
8,auto,592720.11
9,garden_tools,485256.46


In [27]:
# Revenue dan jumlah order selesai per bulan
q("""
SELECT
  STRFTIME('%Y-%m', order_purchase_timestamp) AS month,
  COUNT(DISTINCT o.order_id) AS order_amount,
  ROUND(SUM(op.payment_value), 2) AS revenue
FROM orders o
JOIN order_payments op on o.order_id = op.order_id
WHERE o.order_status LIKE "delivered"
GROUP BY month
ORDER BY month
""")

,month,order_amount,revenue
0,2016-10,265,46566.71
1,2016-12,1,19.62
2,2017-01,750,127545.67
3,2017-02,1653,271298.65
4,2017-03,2546,414369.39
5,2017-04,2303,390952.18
6,2017-05,3546,567066.73
7,2017-06,3135,490225.60
8,2017-07,3872,566403.93
9,2017-08,4193,646000.61
